# Ligamento · S0.9 definitivo (Colab GPU)

Baselines C1 (softmax) + C2 (delta), 8 semillas, con los cierres de rigor: precisión `highest`, XLA determinista, S0.7 re-corrido en la GPU real, checkpoint/reanudación por semilla y **sync a Google Drive** (sobrevive al cierre de sesión).

**Uso:** `Entorno de ejecución → Cambiar tipo de entorno → GPU`, luego `Entorno de ejecución → Ejecutar todo`. Si la sesión se corta, volvé a ejecutar todo: las semillas ya hechas se saltan (están en Drive).

In [ ]:
# 1) Verificar GPU
!nvidia-smi -L || echo 'SIN GPU: Entorno de ejecucion -> Cambiar tipo -> GPU'

In [ ]:
# 2) Dependencias. Colab suele traer jax+GPU; aseguramos optax y jax con CUDA.
!pip -q install optax >/dev/null
import jax; print('jax', jax.__version__, '| devices:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'No hay GPU visible para JAX. Activa GPU en el entorno.'

In [ ]:
# 3) Clonar el repo publico (codigo ya pre-registrado)
![ -d telar-ligamento ] && (cd telar-ligamento && git pull -q) || git clone -q https://github.com/SperanzaMax/telar-ligamento
!ls telar-ligamento/experimentos/fase0

In [ ]:
# 4) Montar Drive para sync fuera de sesion (checkpoints por semilla)
from google.colab import drive
drive.mount('/content/drive')
import os; RESULTS='/content/drive/MyDrive/ligamento_s09'; os.makedirs(RESULTS, exist_ok=True)
print('resultados ->', RESULTS)

In [ ]:
# 5) Correr S0.9 definitivo (C1+C2, 8 semillas). Reanudable: re-ejecutar saltea lo ya hecho.
%env RESULTS_DIR=/content/drive/MyDrive/ligamento_s09
%env N_SEEDS=8
%env STEPS=2500
!cd telar-ligamento/experimentos/fase0 && python fase0_s09.py

In [ ]:
# 6) Ver el resultado agregado (margenes + cargas de evaluacion)
print(open('/content/drive/MyDrive/ligamento_s09/../ligamento_s09/s09_summary.json').read() if os.path.exists('/content/drive/MyDrive/ligamento_s09/s09_summary.json') else 'aun no generado')
import glob; print('\ncheckpoints por semilla:'); print('\n'.join(sorted(glob.glob('/content/drive/MyDrive/ligamento_s09/*.json'))))